# Same-dataset benchmark — YOLO26s + SCConv

Ablation chỉ thay hai `C3k2` ở nhánh neck tạo feature P3/8 và P4/16 bằng `C3k2_SCConv`. Dataset, split, seed, augmentation và toàn bộ protocol giữ nguyên baseline.

**An toàn khi mở:** các cờ `RUN_SCConv_*` và `RUN_COMPARISON` đều mặc định `False`; notebook không train khi được tạo. Folder `test` bị khóa.

In [ ]:
%pip install -q ultralytics==8.4.155 pyyaml pandas matplotlib seaborn


In [ ]:
from pathlib import Path
from collections import Counter
import inspect
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
import ultralytics
from PIL import Image
from ultralytics import YOLO

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Không chạy trong Google Colab; dùng các đường dẫn local nếu có.')

ULTRALYTICS_VERSION = '8.4.155'
MODEL_NAME = 'yolo26s.pt'
EXPECTED_DATASET_NAME = 'No Helmet No Ride'
EXPECTED_CLASS_NAMES = ['helmet', 'nohelmet', 'riders']
EXPECTED_TRAIN_IMAGES, EXPECTED_VALID_IMAGES, EXPECTED_TEST_IMAGES = 3912, 355, 101
IMG_SIZE = 640
RANDOM_SEED = 0
RUN_SCConv_TRAINING = False
RUN_SCConv_VALIDATION = False
RUN_SCConv_COMPLEXITY = False
RUN_COMPARISON = False
DATASET_ROOT_OVERRIDE = None

assert ultralytics.__version__ == ULTRALYTICS_VERSION, ultralytics.__version__
SEARCH_ROOTS = [Path('/content/drive/MyDrive'), Path('/content'), Path.cwd()]
DATASET_CANDIDATE_NAMES = ['NO Helmet NO Ride.v2i.yolov8', 'NO Helmet NO Ride']
def locate_dataset():
    if DATASET_ROOT_OVERRIDE:
        root = Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
        if not (root / 'data.yaml').exists(): raise FileNotFoundError(root / 'data.yaml')
        return root
    for search_root in SEARCH_ROOTS:
        for name in DATASET_CANDIDATE_NAMES:
            for candidate in (search_root / name, search_root / 'DetectedHelmets' / name):
                if (candidate / 'data.yaml').exists(): return candidate.resolve()
    raise FileNotFoundError('Không tìm thấy dataset No Helmet No Ride; đặt DATASET_ROOT_OVERRIDE.')
DATASET_ROOT = locate_dataset(); DATA_YAML_PATH = DATASET_ROOT / 'data.yaml'
with DATA_YAML_PATH.open('r', encoding='utf-8') as handle: DATA_CONFIG = yaml.safe_load(handle)
def normalize_names(raw_names):
    return [raw_names[key] for key in sorted(raw_names, key=lambda item: int(item))] if isinstance(raw_names, dict) else list(raw_names or [])
CLASS_NAMES = normalize_names(DATA_CONFIG.get('names')); assert DATA_CONFIG.get('nc') == 3; assert CLASS_NAMES == EXPECTED_CLASS_NAMES, CLASS_NAMES
def resolve_split_dir(split_name):
    raw_path = Path(str(DATA_CONFIG.get(split_name, '')))
    candidates = [raw_path if raw_path.is_absolute() else DATA_YAML_PATH.parent / raw_path, DATASET_ROOT / split_name / 'images', DATA_YAML_PATH.parent.parent / split_name / 'images']
    for candidate in candidates:
        if candidate.exists(): return candidate.resolve()
    raise FileNotFoundError(f'Không resolve được {split_name}: {candidates}')
SPLIT_DIRS = {split_name: resolve_split_dir(split_name) for split_name in ('train', 'val', 'test')}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def image_files(directory): return sorted(path for path in Path(directory).iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
IMAGE_COUNTS = {name: len(image_files(path)) for name, path in SPLIT_DIRS.items()}
print('Dataset root:', DATASET_ROOT); print('data.yaml:', DATA_YAML_PATH); print('Raw split values:', {key: DATA_CONFIG.get(key) for key in ('train', 'val', 'test')}); print('Resolved split directories:', SPLIT_DIRS); print('Image counts:', IMAGE_COUNTS)
assert IMAGE_COUNTS == {'train': EXPECTED_TRAIN_IMAGES, 'val': EXPECTED_VALID_IMAGES, 'test': EXPECTED_TEST_IMAGES}, IMAGE_COUNTS

OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/paper2025_benchmark')
EXPERIMENT_NAME = 'paper2025_yolo26s_scconv_640_200ep'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'
SCONV_SUMMARY_PATH = OUTPUT_DIR / 'paper2025_yolo26s_scconv_summary.csv'
BASELINE_SUMMARY_PATH = OUTPUT_DIR / 'paper2025_yolo26s_summary.csv'
TEST_EVALUATED = False
TRAIN_CONFIG = {
    'epochs': 200, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4,
    'optimizer': 'auto', 'seed': 0, 'deterministic': True, 'pretrained': True,
    'amp': True, 'patience': 50, 'close_mosaic': 10,
    'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
    'translate': 0.1, 'scale': 0.5, 'fliplr': 0.5, 'mosaic': 1.0,
    'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0,
}
RUNTIME_DATA_YAML_PATH = OUTPUT_DIR / 'resolved_data.yaml'
RUNTIME_DATA_CONFIG = dict(DATA_CONFIG); RUNTIME_DATA_CONFIG.update({'path': str(DATASET_ROOT), 'train': str(SPLIT_DIRS['train']), 'val': str(SPLIT_DIRS['val']), 'test': str(SPLIT_DIRS['test']), 'nc': 3, 'names': CLASS_NAMES})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with RUNTIME_DATA_YAML_PATH.open('w', encoding='utf-8') as handle: yaml.safe_dump(RUNTIME_DATA_CONFIG, handle, sort_keys=False, allow_unicode=True)
print('Ultralytics:', ultralytics.__version__); print('Classes:', CLASS_NAMES); print('Protocol:', TRAIN_CONFIG)

## Audit label và dry-run detection loader

Polygon source được giữ nguyên. Ultralytics detection loader phải đọc được và derive bbox trong bộ nhớ; nếu không, notebook dừng trước khi dựng model.

In [ ]:
label_files = sorted((SPLIT_DIRS['train'].parent / 'labels').glob('*.txt'))
sample_files = random.Random(RANDOM_SEED).sample(label_files, min(1000, len(label_files)))
row_length_counts = Counter()
for label_path in sample_files:
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if line: row_length_counts[len(line.split())] += 1
if not row_length_counts: raise RuntimeError('Không có label row để audit.')
HAS_POLYGON_LABELS = any(length > 5 for length in row_length_counts)
if any(length not in (5,) and length <= 5 for length in row_length_counts): raise RuntimeError(f'Row label không hợp lệ: {row_length_counts}')
LABEL_MODE = 'polygon source, detection boxes derived by Ultralytics' if HAS_POLYGON_LABELS else 'YOLO detection rows'
print('Scanned label files:', len(sample_files)); print('Detection label rows:', row_length_counts.get(5, 0)); print('Polygon label rows:', sum(value for key, value in row_length_counts.items() if key > 5)); print('LABEL_MODE =', LABEL_MODE)

from ultralytics.data.dataset import YOLODataset
def detection_loader_dry_run(image_dir):
    signature = inspect.signature(YOLODataset); optional = {'imgsz': IMG_SIZE, 'cache': False, 'augment': False, 'rect': False, 'stride': 32, 'pad': 0.0, 'single_cls': False, 'classes': None, 'fraction': 1.0, 'prefix': 'audit: '}
    accepted = {key: value for key, value in optional.items() if key in signature.parameters}
    dataset = YOLODataset(str(image_dir), data={'nc': 3, 'names': CLASS_NAMES}, task='detect', **accepted)
    if len(dataset) == 0: raise RuntimeError(f'Loader không thấy ảnh tại {image_dir}')
    item = dataset[0]
    bbox_count = len(item['bboxes']) if 'bboxes' in item else len(item['instances'].bboxes) if 'instances' in item else 0
    if bbox_count == 0: raise RuntimeError('Dry-run loader không trả bounding boxes.')
    return len(dataset), bbox_count
train_loader_len, train_bbox_count = detection_loader_dry_run(SPLIT_DIRS['train']); valid_loader_len, valid_bbox_count = detection_loader_dry_run(SPLIT_DIRS['val'])
print('Detection loader dry-run:', {'train': train_loader_len, 'valid': valid_loader_len, 'train_sample_boxes': train_bbox_count, 'valid_sample_boxes': valid_bbox_count})
print('No polygon-to-bbox files were written; dataset remains unchanged.')

## Runtime graph, SCConv và đúng hai target C3k2

Target được chọn từ runtime graph: đúng một C3k2 là input Detect ở stride 8 và đúng một ở stride 16, nằm trong neck sau upsample. Không đoán index cố định.

In [ ]:
base_model = YOLO(MODEL_NAME)
def module_count(module, class_name): return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)
def probe_c3k2(yolo_model, imgsz=640):
    records, handles = [], []
    def make_hook(index, module):
        def hook(_module, inputs, output):
            if torch.is_tensor(output):
                input_value = inputs[0] if inputs else None; records.append({'index': index, 'from': getattr(module, 'f', None), 'input_shape': tuple(input_value.shape) if torch.is_tensor(input_value) else None, 'output_shape': tuple(output.shape), 'stride': imgsz // output.shape[-1]})
        return hook
    for index, module in enumerate(yolo_model.model.model):
        if module.__class__.__name__ == 'C3k2': handles.append(module.register_forward_hook(make_hook(index, module)))
    device = next(yolo_model.model.parameters()).device; was_training = yolo_model.model.training; yolo_model.model.eval()
    with torch.no_grad(): yolo_model.model(torch.zeros(1, 3, imgsz, imgsz, device=device))
    if was_training: yolo_model.model.train()
    for handle in handles: handle.remove()
    return sorted(records, key=lambda item: item['index'])
c3k2_probe = probe_c3k2(base_model); detect_entries = [(i, m) for i, m in enumerate(base_model.model.model) if m.__class__.__name__ in {'Detect', 'WorldDetect', 'YOLOEDetect'}]
if not detect_entries: print(base_model.model); raise RuntimeError('Detect head not found.')
detect_index, detect_head = min(detect_entries, key=lambda item: item[0]); detect_from = getattr(detect_head, 'f', None)
if not isinstance(detect_from, (list, tuple)): print(base_model.model); raise RuntimeError('Detect input indices unavailable.')
detect_source_indices = {int(index) for index in detect_from}; neck_start = min(i for i, m in enumerate(base_model.model.model) if m.__class__.__name__ == 'Upsample' and i < detect_index)
neck_candidates = [record for record in c3k2_probe if neck_start < record['index'] < detect_index and record['index'] in detect_source_indices and record['stride'] in (8, 16)]
selected_by_stride = {}
for stride in (8, 16):
    matches = [record for record in neck_candidates if record['stride'] == stride]
    if len(matches) != 1: print('Candidates:', matches); print(base_model.model); raise RuntimeError(f'Expected exactly one C3k2 target at stride {stride}.')
    selected_by_stride[stride] = matches[0]
c3k2_specs = [{'level': f'P{stride // 8 + 2}/{stride}', **selected_by_stride[stride]} for stride in (8, 16)]
print('Selected targets:', c3k2_specs); assert len(c3k2_specs) == 2 and {item['stride'] for item in c3k2_specs} == {8, 16}

def nearest_valid_group_num(c_num, requested):
    valid = [group for group in range(1, c_num + 1) if c_num % group == 0]; return min(valid, key=lambda group: (abs(group - requested), -group))
class GroupBatchnorm2d(nn.Module):
    def __init__(self, c_num, group_num=4, eps=1e-5):
        super().__init__(); self.group_num = nearest_valid_group_num(c_num, group_num); self.weight = nn.Parameter(torch.ones(c_num, 1, 1)); self.bias = nn.Parameter(torch.zeros(c_num, 1, 1)); self.eps = eps
    def forward(self, x):
        batch, channels, height, width = x.shape; grouped = x.reshape(batch, self.group_num, -1); mean = grouped.mean(2, keepdim=True); variance = grouped.var(2, keepdim=True, unbiased=False); normalized = ((grouped - mean) / torch.sqrt(variance + self.eps)).reshape(batch, channels, height, width); return normalized * self.weight + self.bias
class SRU(nn.Module):
    def __init__(self, oup_channels, group_num=4, gate_threshold=0.5):
        super().__init__(); self.gn = GroupBatchnorm2d(oup_channels, group_num); self.gate_threshold = gate_threshold
        if oup_channels % 2: raise ValueError('SRU cross reconstruction requires even channels.')
    def forward(self, x):
        gn_x = self.gn(x); gamma = self.gn.weight; gamma_weight = (gamma / (gamma.sum() + torch.finfo(gamma.dtype).eps)).view(1, -1, 1, 1); weights = torch.sigmoid(gn_x * gamma_weight); informative = torch.where(weights > self.gate_threshold, torch.ones_like(weights), weights); non_informative = torch.where(weights > self.gate_threshold, torch.zeros_like(weights), weights); half = x.shape[1] // 2; x11, x12 = torch.split(informative * x, [half, half], 1); x21, x22 = torch.split(non_informative * x, [half, half], 1); return torch.cat([x11 + x22, x12 + x21], 1)
class CRU(nn.Module):
    def __init__(self, op_channel, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__(); self.up_channel = int(alpha * op_channel); self.low_channel = op_channel - self.up_channel; self.up_squeeze = self.up_channel // squeeze_ratio; self.low_squeeze = self.low_channel // squeeze_ratio
        if min(self.up_squeeze, self.low_squeeze) < 1 or self.up_squeeze % group_size or op_channel % group_size: raise ValueError('CRU channel configuration is incompatible.')
        self.squeeze1 = nn.Conv2d(self.up_channel, self.up_squeeze, 1, bias=False); self.squeeze2 = nn.Conv2d(self.low_channel, self.low_squeeze, 1, bias=False); self.GWC = nn.Conv2d(self.up_squeeze, op_channel, group_kernel_size, 1, group_kernel_size // 2, groups=group_size); self.PWC1 = nn.Conv2d(self.up_squeeze, op_channel, 1, bias=False); self.PWC2 = nn.Conv2d(self.low_squeeze, op_channel - self.low_squeeze, 1, bias=False); self.advavg = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        upper, lower = torch.split(x, [self.up_channel, self.low_channel], 1); upper, lower = self.squeeze1(upper), self.squeeze2(lower); upper_branch = self.GWC(upper) + self.PWC1(upper); lower_branch = torch.cat([self.PWC2(lower), lower], 1); fused = torch.cat([upper_branch, lower_branch], 1); fused = F.softmax(self.advavg(fused), dim=1) * fused; first, second = torch.split(fused, fused.shape[1] // 2, 1); return first + second
class ScConv(nn.Module):
    def __init__(self, op_channel, group_num=4, gate_threshold=0.5, alpha=0.5, squeeze_ratio=2, group_size=2, group_kernel_size=3):
        super().__init__(); self.SRU = SRU(op_channel, group_num, gate_threshold); self.CRU = CRU(op_channel, alpha, squeeze_ratio, group_size, group_kernel_size)
    def forward(self, x): return self.CRU(self.SRU(x))
unit_input = torch.randn(2, 32, 16, 16, requires_grad=True); unit_module = ScConv(32); unit_output = unit_module(unit_input); assert unit_output.shape == unit_input.shape; unit_output.mean().backward(); assert any(parameter.grad is not None for parameter in unit_module.parameters()); print('SCConv unit shape/gradient checks: OK')

## C3k2_SCConv integration và preflight

In [ ]:
SCONV_CONFIG = {'group_num': 4, 'gate_threshold': 0.5, 'alpha': 0.5, 'squeeze_ratio': 2, 'group_size': 2, 'group_kernel_size': 3}
def wrapped_channels(wrapper):
    raw = getattr(wrapper, 'conv', wrapper); return int(raw.in_channels), int(raw.out_channels)
def wrapped_stride(wrapper):
    stride = getattr(getattr(wrapper, 'conv', wrapper), 'stride', 1); return (stride, stride) if isinstance(stride, int) else tuple(stride)
class Bottleneck_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__(); self.cv1 = original.cv1; self.add = bool(getattr(original, 'add', False)); input_channels, output_channels = wrapped_channels(original.cv2); self.scconv_applied = wrapped_stride(original.cv2) == (1, 1) and input_channels == output_channels and input_channels % 2 == 0; self.cv2 = ScConv(input_channels, **SCONV_CONFIG) if self.scconv_applied else original.cv2
    def forward(self, x):
        output = self.cv2(self.cv1(x)); return x + output if self.add else output
def replace_bottlenecks(module, stats):
    if module.__class__.__name__ == 'Bottleneck':
        replacement = Bottleneck_SCConv(module); stats['seen'] += 1; stats['scconv'] += int(replacement.scconv_applied); stats['kept'] += int(not replacement.scconv_applied); return replacement
    for name, child in list(module.named_children()):
        replacement = replace_bottlenecks(child, stats)
        if replacement is not child: setattr(module, name, replacement)
    return module
class C3k2_SCConv(nn.Module):
    def __init__(self, original):
        super().__init__(); self.c, self.cv1, self.cv2 = original.c, original.cv1, original.cv2; self.stats = {'seen': 0, 'scconv': 0, 'kept': 0}; self.m = replace_bottlenecks(original.m, self.stats)
    def forward(self, x):
        values = list(self.cv1(x).chunk(2, 1)); values.extend(block(values[-1]) for block in self.m); return self.cv2(torch.cat(values, 1))
    def forward_split(self, x):
        values = list(self.cv1(x).split((self.c, self.c), 1)); values.extend(block(values[-1]) for block in self.m); return self.cv2(torch.cat(values, 1))
def copy_graph_metadata(new, old, index, from_value):
    new.i = getattr(old, 'i', index); new.f = getattr(old, 'f', from_value); new.type = getattr(old, 'type', new.__class__.__name__); new.np = sum(parameter.numel() for parameter in new.parameters()); return new
def count_class(module, class_name): return sum(1 for item in module.modules() if item.__class__.__name__ == class_name)
def build_custom_model():
    model = YOLO(MODEL_NAME); replacement_stats = []
    for spec in c3k2_specs:
        old = model.model.model[spec['index']]
        if old.__class__.__name__ != 'C3k2': raise RuntimeError(f'Target at {spec["index"]} is not C3k2.')
        new = C3k2_SCConv(old); replacement_stats.append((spec['level'], new.stats)); model.model.model[spec['index']] = copy_graph_metadata(new, old, spec['index'], spec['from'])
    source_state, target_state = base_model.model.state_dict(), model.model.state_dict(); compatible = {key: value for key, value in source_state.items() if key in target_state and tuple(value.shape) == tuple(target_state[key].shape)}; report = model.model.load_state_dict(compatible, strict=False)
    print('Replacement stats:', replacement_stats); print('Pretrained tensors transferred:', len(compatible), 'missing:', len(report.missing_keys)); assert count_class(model.model, 'C3k2_SCConv') == 2 and count_class(model.model, 'ScConv') > 0; return model
custom_model = build_custom_model(); custom_pytorch_model = custom_model.model; device = next(custom_pytorch_model.parameters()).device
custom_pytorch_model.eval(); shape_records, handles = {}, []
for spec in c3k2_specs:
    index = spec['index']; handles.append(custom_pytorch_model.model[index].register_forward_hook(lambda _module, inputs, output, index=index: shape_records.__setitem__(index, (tuple(inputs[0].shape), tuple(output.shape)))))
with torch.no_grad(): dummy_output = custom_pytorch_model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device))
for handle in handles: handle.remove()
for spec in c3k2_specs:
    observed_input, observed_output = shape_records[spec['index']]; assert observed_input[1:] == spec['input_shape'][1:]; assert observed_output[1:] == spec['output_shape'][1:]
custom_pytorch_model.train(); custom_pytorch_model.zero_grad(set_to_none=True); graph_output = custom_pytorch_model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device, requires_grad=True))
def sum_tensors(value):
    if torch.is_tensor(value): return value.float().sum()
    if isinstance(value, (list, tuple)): return sum((sum_tensors(item) for item in value), torch.zeros((), device=device))
    if isinstance(value, dict): return sum((sum_tensors(item) for item in value.values()), torch.zeros((), device=device))
    return torch.zeros((), device=device)
sum_tensors(graph_output).backward(); scconv_grad_ok = any(parameter.grad is not None for item in custom_pytorch_model.modules() if item.__class__.__name__ == 'ScConv' for parameter in item.parameters()); custom_pytorch_model.eval(); assert scconv_grad_ok
print('C3k2_SCConv count:', count_class(custom_pytorch_model, 'C3k2_SCConv')); print('ScConv count:', count_class(custom_pytorch_model, 'ScConv')); print('Target shapes:', shape_records); print('SCConv gradient OK:', scconv_grad_ok)

## SCConv training — chỉ chạy khi chủ động bật cờ

Custom trainer giữ nguyên `custom_pytorch_model` để Ultralytics không dựng lại YOLO26s baseline và làm mất SCConv.

In [ ]:
from ultralytics.models.yolo.detect import DetectionTrainer
class CustomSCConvTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True): return custom_pytorch_model
    def setup_model(self):
        self.model = custom_pytorch_model; assert count_class(self.model, 'C3k2_SCConv') == 2; print('C3k2_SCConv INSIDE setup_model:', count_class(self.model, 'C3k2_SCConv')); return None
TRAIN_OVERRIDES = {'model': MODEL_NAME, 'data': str(RUNTIME_DATA_YAML_PATH), 'project': str(OUTPUT_DIR), 'name': EXPERIMENT_NAME, 'exist_ok': False, 'save': True, 'plots': True, **TRAIN_CONFIG}
trainer = None; c3k2_scconv_before_train = count_class(custom_pytorch_model, 'C3k2_SCConv'); c3k2_scconv_inside_trainer = None
if RUN_SCConv_TRAINING:
    trainer = CustomSCConvTrainer(overrides=TRAIN_OVERRIDES, _callbacks=custom_model.callbacks); trainer.model = custom_pytorch_model; c3k2_scconv_inside_trainer = count_class(trainer.model, 'C3k2_SCConv'); assert c3k2_scconv_inside_trainer == c3k2_scconv_before_train == 2
    train_results = trainer.train()
    if not BEST_MODEL_PATH.exists(): raise FileNotFoundError(BEST_MODEL_PATH)
    (EXPERIMENT_DIR / 'ultralytics_version.txt').write_text('ultralytics==' + ultralytics.__version__ + '\n' + 'model=' + MODEL_NAME + '\n' + 'c3k2_scconv_expected_count=2\n', encoding='utf-8')
    print('Training klaar:', BEST_MODEL_PATH)
else:
    print('TRAINING DISABLED — audit-only notebook. Zet RUN_SCConv_TRAINING=True om sau này train.')

## Validation, per-class metrics và confusion matrix — chỉ `valid`

In [ ]:
def checkpoint_state(path):
    try: payload = torch.load(str(path), map_location='cpu', weights_only=False)
    except TypeError: payload = torch.load(str(path), map_location='cpu')
    if isinstance(payload, nn.Module): return payload.state_dict()
    if isinstance(payload, dict):
        for key in ('ema', 'model', 'state_dict'):
            value = payload.get(key)
            if isinstance(value, nn.Module): return value.state_dict()
            if isinstance(value, dict):
                tensors = {name: item for name, item in value.items() if torch.is_tensor(item)}
                if tensors: return tensors
        tensors = {name: item for name, item in payload.items() if torch.is_tensor(item)}
        if tensors: return tensors
    raise TypeError('Không tìm thấy state dict trong checkpoint.')
def load_custom_checkpoint(path):
    rebuilt = build_custom_model(); state = checkpoint_state(path)
    if state and all(name.startswith('module.') for name in state): state = {name[7:]: value for name, value in state.items()}
    target = rebuilt.model.state_dict(); compatible = {name: value for name, value in state.items() if name in target and tuple(value.shape) == tuple(target[name].shape)}
    if not compatible: raise RuntimeError('Checkpoint không có tensor tương thích với C3k2_SCConv.')
    report = rebuilt.model.load_state_dict(compatible, strict=False); print('Checkpoint tensors loaded:', len(compatible), 'missing:', len(report.missing_keys)); assert count_class(rebuilt.model, 'C3k2_SCConv') == 2; return rebuilt
def safe_f1(precision, recall): return 2 * precision * recall / (precision + recall) if precision + recall else 0.0
def collect_metrics(result):
    overall = {'Precision': float(result.box.mp), 'Recall': float(result.box.mr), 'mAP50': float(result.box.map50), 'mAP50_95': float(result.box.map)}; overall['F1'] = safe_f1(overall['Precision'], overall['Recall']); per_class = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision, recall, ap50, ap5095 = map(float, result.box.class_result(class_id)); per_class[class_name] = {'Precision': precision, 'Recall': recall, 'F1': safe_f1(precision, recall), 'AP50': ap50, 'AP50-95': ap5095}
    return overall, per_class
validation_overall = validation_per_class = None; best_model = None; test_metrics = None; test_summary = None
if RUN_SCConv_VALIDATION:
    if not BEST_MODEL_PATH.exists(): raise FileNotFoundError(BEST_MODEL_PATH)
    best_model = load_custom_checkpoint(BEST_MODEL_PATH)
    validation_result = best_model.val(data=str(RUNTIME_DATA_YAML_PATH), split='val', imgsz=IMG_SIZE, device=0, workers=4, plots=True, verbose=False)
    validation_overall, validation_per_class = collect_metrics(validation_result); display(pd.DataFrame(validation_per_class).T); print('Overall validation:', validation_overall); print('Confusion matrix:', EXPERIMENT_DIR / 'confusion_matrix.png')
else:
    print('VALIDATION DISABLED — chạy sau khi có best.pt và đặt RUN_SCConv_VALIDATION=True.')

## Complexity và inference speed trên tối đa 50 ảnh validation

In [ ]:
complexity_metrics = {'Parameters': np.nan, 'GFLOPs': np.nan, 'Inference_ms': np.nan, 'FPS': np.nan, 'GPU_peak_VRAM_MB': np.nan}
if RUN_SCConv_COMPLEXITY:
    if best_model is None: raise RuntimeError('Cần chạy validation trước complexity.')
    info = best_model.info(verbose=False, imgsz=IMG_SIZE); gflops = np.nan
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                gflops = float(info[key]); break
    elif isinstance(info, (list, tuple)):
        for value in reversed(info):
            if isinstance(value, (int, float)) and float(value) > 0: gflops = float(value); break
    speed_images = image_files(SPLIT_DIRS['val'])[:50]; assert speed_images
    for image_path in speed_images[:min(2, len(speed_images))]: best_model.predict(source=str(image_path), imgsz=IMG_SIZE, device=0, save=False, verbose=False)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    elapsed = []
    for image_path in speed_images:
        if torch.cuda.is_available(): torch.cuda.synchronize()
        started = time.perf_counter(); best_model.predict(source=str(image_path), imgsz=IMG_SIZE, device=0, save=False, verbose=False)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        elapsed.append(time.perf_counter() - started)
    inference_ms = float(np.mean(elapsed) * 1000); peak_vram = float(torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else np.nan
    complexity_metrics = {'Parameters': sum(parameter.numel() for parameter in best_model.model.parameters()), 'GFLOPs': gflops, 'Inference_ms': inference_ms, 'FPS': 1000.0 / inference_ms, 'GPU_peak_VRAM_MB': peak_vram}; print('Complexity:', complexity_metrics)
else:
    print('COMPLEXITY DISABLED — đặt RUN_SCConv_COMPLEXITY=True sau validation nếu cần đo.')

## Save summary và same-protocol comparison với paper 2025

Comparison chỉ chạy sau khi baseline và SCConv summary đã được tạo. Metric được chuẩn hóa từ decimal của Ultralytics sang phần trăm; delta dùng **percentage points**. Không so sánh paper self-collected test vì không có tập test đó.

In [ ]:
if RUN_SCConv_VALIDATION:
    per_class = validation_per_class
    row = {'Model': 'YOLO26s+SCConv', 'Dataset': EXPECTED_DATASET_NAME, 'TrainImages': IMAGE_COUNTS['train'], 'ValidImages': IMAGE_COUNTS['val'], 'imgsz': IMG_SIZE, 'epochs': TRAIN_CONFIG['epochs'], **validation_overall, 'Helmet_AP50': per_class['helmet']['AP50'], 'NoHelmet_AP50': per_class['nohelmet']['AP50'], 'Riders_AP50': per_class['riders']['AP50'], **complexity_metrics}
    scconv_summary = pd.DataFrame([row]); OUTPUT_DIR.mkdir(parents=True, exist_ok=True); scconv_summary.to_csv(SCONV_SUMMARY_PATH, index=False); display(scconv_summary); print('Saved:', SCONV_SUMMARY_PATH)
else:
    print('SCConv summary chưa ghi vì RUN_SCConv_VALIDATION=False.')

if RUN_COMPARISON:
    if not BASELINE_SUMMARY_PATH.exists() or not SCONV_SUMMARY_PATH.exists(): raise FileNotFoundError('Cần chạy cả baseline và SCConv để tạo comparison.')
    baseline = pd.read_csv(BASELINE_SUMMARY_PATH).iloc[0]; ours = pd.read_csv(SCONV_SUMMARY_PATH).iloc[0]
    paper_rows = [
        {'Model': 'YOLO11n', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 89.8, 'mAP50-95': 60.4},
        {'Model': 'YOLO11s', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 88.6, 'mAP50-95': 61.4},
        {'Model': 'YOLO11m', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 87.6, 'mAP50-95': 62.5},
        {'Model': 'YOLO11l', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 87.5, 'mAP50-95': 61.9},
        {'Model': 'YOLO11x', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 88.6, 'mAP50-95': 60.9},
        {'Model': 'YOLOv8m', 'Source': 'Paper 2025', 'Dataset': EXPECTED_DATASET_NAME, 'imgsz': 640, 'Epoch': 200, 'mAP50': 87.7, 'mAP50-95': 61.5},
        {'Model': 'YOLO26s', 'Source': 'Our reproduction', 'Dataset': 'same source dataset', 'imgsz': 640, 'Epoch': 200, 'mAP50': float(baseline['mAP50']) * 100, 'mAP50-95': float(baseline['mAP50_95']) * 100, 'Precision': float(baseline['Precision']) * 100, 'Recall': float(baseline['Recall']) * 100, 'F1': float(baseline['F1']) * 100},
        {'Model': 'YOLO26s + SCConv', 'Source': 'Our proposed architecture benchmark', 'Dataset': 'same source dataset', 'imgsz': 640, 'Epoch': 200, 'mAP50': float(ours['mAP50']) * 100, 'mAP50-95': float(ours['mAP50_95']) * 100, 'Precision': float(ours['Precision']) * 100, 'Recall': float(ours['Recall']) * 100, 'F1': float(ours['F1']) * 100},
    ]
    comparison = pd.DataFrame(paper_rows); comparison['Delta_vs_YOLO11n_mAP50_pp'] = comparison['mAP50'] - 89.8; comparison['Delta_vs_best_paper_mAP50_95_pp'] = comparison['mAP50-95'] - 62.5
    comparison_path = OUTPUT_DIR / 'paper2025_benchmark_comparison.csv'; comparison.to_csv(comparison_path, index=False); display(comparison); print('Saved:', comparison_path)
    markdown_lines = ['| Model | mAP50 | mAP50-95 | Δ mAP50 vs YOLO11n |', '|---|---:|---:|---:|']
    for _, item in comparison.iterrows():
        model_label = f"**{item['Model']}**" if item['Model'] in {'YOLO11n', 'YOLO11m', 'YOLO26s', 'YOLO26s + SCConv'} else item['Model']; markdown_lines.append(f"| {model_label} | {item['mAP50']:.2f}% | {item['mAP50-95']:.2f}% | {item['Delta_vs_YOLO11n_mAP50_pp']:+.2f} pp |")
    markdown_lines += ['', 'Direct comparison is limited to the public training/validation dataset protocol. The paper\'s self-collected test set is not available, therefore its reported test metrics are not treated as a directly reproducible benchmark.']
    (OUTPUT_DIR / 'paper2025_comparison_table.md').write_text('\n'.join(markdown_lines), encoding='utf-8'); print('Saved:', OUTPUT_DIR / 'paper2025_comparison_table.md')
else:
    print('COMPARISON DISABLED — bật RUN_COMPARISON=True sau khi có cả hai summary.')

In [ ]:
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert DATASET_ROOT.name.lower().startswith('no helmet no ride')
assert IMAGE_COUNTS == {'train': 3912, 'val': 355, 'test': 101}
assert CLASS_NAMES == ['helmet', 'nohelmet', 'riders']
assert TRAIN_CONFIG['epochs'] == 200 and TRAIN_CONFIG['imgsz'] == 640 and TRAIN_CONFIG['batch'] == 16 and TRAIN_CONFIG['seed'] == 0 and TRAIN_CONFIG['deterministic'] is True
assert len(c3k2_specs) == 2 and {item['stride'] for item in c3k2_specs} == {8, 16}
assert not TEST_EVALUATED
assert count_class(custom_pytorch_model, 'C3k2_SCConv') == 2
forbidden = {'P2', 'DySample', 'iAFF', 'MultiSEAM', 'HardMining'}; assert not ({item.__class__.__name__ for item in custom_pytorch_model.modules()} & forbidden)
print('Final audit: PASS'); print('Dataset audit included: YES | 3912/355 strict guard: YES | Test locked: YES | Same-protocol comparison: YES'); print('Training executed in this creation session: NO')